# Modelo de Classificacao de Potencia - Inibidores de Cruzipain

**Alvo molecular:** Cruzipain (Cruzain) | CHEMBL3563 | *Trypanosoma cruzi*

Este notebook desenvolve um classificador multiclasse capaz de estimar a potencia de um inibidor de Cruzipain a partir de seus descritores moleculares. A potencia e dividida em tres niveis com base no valor de IC50:

| Classe | Faixa de IC50 | Interpretacao |
|---|---|---|
| 0 | menor ou igual a Q25 | Potente |
| 1 | entre Q25 e Q50 | Moderado |
| 2 | maior que Q50 | Fraco |

Os dados de entrada (`inhibitors_pruned.csv`) sao produzidos no notebook `02_descritores`, que calcula 1875 descritores moleculares via PaDEL e aplica selecao de atributos, resultando em 628 features.

**Metodologia:** quatro algoritmos sao comparados (Random Forest, HistGradientBoosting, SVM e XGBoost), avaliados por validacao cruzada de 10 folds. O melhor modelo passa por otimizacao de hiperparametros.

## 1. Importacao de bibliotecas e carregamento dos dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle, os
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     StratifiedKFold, RandomizedSearchCV)
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay,
                             roc_curve, auc)
from scipy.stats import kruskal
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv(r'..\data\processed\inhibitors_pruned.csv')
print(f"Inibidores: {df.shape}")
print(df['potency_class'].value_counts().sort_index())

## 2. Preparacao da matriz de atributos

A variavel alvo e `potency_class`. As colunas `smiles` e `IC50_nM` sao removidas: o SMILES e um identificador textual e o IC50 define diretamente a classe, de modo que mante-lo como atributo causaria vazamento de informacao (data leakage).

Como os descritores moleculares incluem medidas tridimensionais com grande amplitude de valores, aplica-se um tratamento numerico para garantir estabilidade: limitacao por percentis (1 e 99) para conter valores extremos, recorte a um limite seguro para precisao float32 e imputacao de eventuais valores ausentes pela mediana.

In [ ]:
META = ['smiles', 'IC50_nM', 'potency_class']
X = df.drop(columns=META).copy()
y = df['potency_class']

# Tratamento numerico para estabilidade
X = X.replace([np.inf, -np.inf], np.nan)
for col in X.columns:
    X[col] = X[col].clip(X[col].quantile(0.01), X[col].quantile(0.99))
X = X.clip(-1e6, 1e6)
X = X.fillna(X.median())

print(f"Atributos: {X.shape[1]}")
print(f"Valores infinitos: {np.isinf(X.values).sum()}")
print(f"Valores ausentes: {X.isnull().sum().sum()}")
print(f"Classes: {sorted(y.unique())}")

## 3. Distribuicao das classes

A distribuicao mostra o balanceamento entre os tres niveis de potencia. A classe 2 (inibidores fracos) concentra cerca de metade das amostras, refletindo a divisao por quantis (25% / 25% / 50%).

In [ ]:
counts = y.value_counts().sort_index()
plt.figure(figsize=(8, 6))
plt.bar(counts.index, counts.values, color=['#1D9E75', '#378ADD', '#E24B4A'])
plt.ylabel('Contagem', fontweight='bold')
plt.xlabel('Classe de potencia', fontweight='bold')
plt.xticks([0, 1, 2], ['0 - Potente', '1 - Moderado', '2 - Fraco'])
plt.title('Distribuicao das classes de potencia')
for i, v in zip(counts.index, counts.values):
    plt.text(i, v + 1, str(v), ha='center', fontweight='bold')
os.makedirs('../data/figures', exist_ok=True)
plt.savefig(r'..\data\figures\model2_class_division.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Divisao treino e teste

Particao estratificada 80/20, preservando a proporcao das classes em ambos os conjuntos.

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Treino: {x_train.shape} | Teste: {x_test.shape}")

## 5. Treinamento dos modelos

Quatro algoritmos sao treinados com a mesma particao:

- **Random Forest** - ensemble de arvores por bagging
- **HistGradientBoosting** - boosting baseado em histogramas
- **SVM (RBF)** - maquina de vetores de suporte com kernel radial
- **XGBoost** - gradient boosting otimizado

O SVM e encapsulado em um pipeline com `StandardScaler`, pois e sensivel a escala dos atributos. Sem padronizacao, o kernel RBF e dominado pelos descritores de maior magnitude e o modelo tende a prever apenas a classe majoritaria. Os modelos baseados em arvore sao invariantes a escala e dispensam esse passo.

In [ ]:
clf1 = RandomForestClassifier(n_jobs=-1, n_estimators=150, max_features=50, random_state=42)
clf2 = HistGradientBoostingClassifier(max_iter=262, learning_rate=0.12, min_samples_leaf=40,
                                      l2_regularization=8, max_bins=255, random_state=42)
clf3 = make_pipeline(StandardScaler(),
                     SVC(C=400, kernel='rbf', gamma='scale', probability=True, random_state=42))
clf4 = XGBClassifier(random_state=42)

for clf in [clf1, clf2, clf3, clf4]:
    clf.fit(x_train, y_train)

y_pred  = clf1.predict(x_test)
y_pred2 = clf2.predict(x_test)
y_pred3 = clf3.predict(x_test)
y_pred4 = clf4.predict(x_test)

print(f"Random Forest:        {accuracy_score(y_test, y_pred):.4f}")
print(f"HistGradientBoosting: {accuracy_score(y_test, y_pred2):.4f}")
print(f"SVM (RBF):            {accuracy_score(y_test, y_pred3):.4f}")
print(f"XGBoost:              {accuracy_score(y_test, y_pred4):.4f}")

## 6. Metricas comparativas

Alem da acuracia, avaliam-se precision, recall e F1 ponderados, que consideram o desempenho em cada classe ajustado pelo seu tamanho - relevante dado o desbalanceamento.

In [ ]:
models  = ['RF', 'HGB', 'SVM', 'XGB']
y_preds = [y_pred, y_pred2, y_pred3, y_pred4]

rows = []
for name, yp in zip(models, y_preds):
    rows.append({
        'Modelo': name,
        'Accuracy':  round(accuracy_score(y_test, yp), 4),
        'Precision': round(precision_score(y_test, yp, average='weighted'), 4),
        'Recall':    round(recall_score(y_test, yp, average='weighted'), 4),
        'F1':        round(f1_score(y_test, yp, average='weighted'), 4)
    })
metrics_df = pd.DataFrame(rows)
print(metrics_df.to_string(index=False))

# Grafico de barras agrupadas
xpos = np.arange(len(models))
plt.figure(figsize=(10, 6))
plt.bar(xpos - 0.2, metrics_df['Precision'], width=0.2, label='Precision', color='#cc5000')
plt.bar(xpos,       metrics_df['Recall'],    width=0.2, label='Recall',    color='#9bc209')
plt.bar(xpos + 0.2, metrics_df['F1'],        width=0.2, label='F1',        color='#fcc200')
plt.xticks(xpos, models)
plt.ylabel('Score', fontweight='bold')
plt.title('Precision, Recall e F1 por modelo')
plt.legend()
plt.savefig(r'..\data\figures\model2_prf.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Matrizes de confusao

As matrizes detalham os acertos e erros por classe. A diagonal principal representa as classificacoes corretas. Erros concentrados entre classes adjacentes (0-1 ou 1-2) sao esperados, pois a fronteira de potencia entre niveis vizinhos e naturalmente difusa.

In [ ]:
for yp, name in zip(y_preds, models):
    cm = confusion_matrix(y_test, yp)
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay(cm, display_labels=[0, 1, 2]).plot(cmap='viridis', ax=ax, values_format='d')
    ax.set_title(f'Matriz de confusao - {name}', fontweight='bold')
    plt.savefig(f'../data/figures/model2_cm_{name}.png', dpi=300, bbox_inches='tight')
    plt.show()

## 8. Curvas ROC multiclasse

Para problemas com mais de duas classes utiliza-se a estrategia one-vs-rest: cada classe e avaliada contra as demais e calcula-se a media das curvas. A area sob a curva (AUC) resume a capacidade de discriminacao - quanto mais proxima de 1, melhor.

In [ ]:
classes = [0, 1, 2]
y_test_bin = label_binarize(y_test, classes=classes)
colors = ['#9bc209', '#cc5000', '#00b7eb', '#fcc200']
clfs = [clf1, clf2, clf3, clf4]

plt.figure(figsize=(8, 8))
for clf, name, color in zip(clfs, models, colors):
    proba = clf.predict_proba(x_test)
    fpr, tpr = dict(), dict()
    for c in range(3):
        fpr[c], tpr[c], _ = roc_curve(y_test_bin[:, c], proba[:, c])
    all_fpr = np.unique(np.concatenate([fpr[c] for c in range(3)]))
    mean_tpr = np.zeros_like(all_fpr)
    for c in range(3):
        mean_tpr += np.interp(all_fpr, fpr[c], tpr[c])
    mean_tpr /= 3
    plt.plot(all_fpr, mean_tpr, color=color, linewidth=2.5,
             label=f'{name} (AUC = {auc(all_fpr, mean_tpr):.2f})')
plt.plot([0, 1], [0, 1], 'k--', label='Aleatorio')
plt.xlabel('Taxa de falsos positivos', fontweight='bold')
plt.ylabel('Taxa de verdadeiros positivos', fontweight='bold')
plt.title('Curvas ROC multiclasse (one-vs-rest)')
plt.legend()
plt.savefig(r'..\data\figures\model2_roc.png', dpi=300, bbox_inches='tight')
plt.show()

## 9. Validacao cruzada

A particao unica de teste contem cerca de 100 amostras, o que torna a estimativa de desempenho sensivel ao sorteio especifico. A validacao cruzada de 10 folds fornece uma medida mais robusta, pois avalia o modelo em multiplas particoes e reporta media e desvio padrao.

In [ ]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
rows = []
for name, clf in zip(models, clfs):
    sc = cross_val_score(clf, X, y, cv=cv, scoring='accuracy')
    rows.append({'Modelo': name,
                 'CV Media': round(sc.mean(), 4),
                 'CV Desvio': round(sc.std(), 4)})
cv_df = pd.DataFrame(rows)
print(cv_df.to_string(index=False))

## 10. Otimizacao de hiperparametros

Os hiperparametros iniciais sao valores de referencia. Como cada conjunto de dados possui caracteristicas proprias, realiza-se uma busca aleatoria (`RandomizedSearchCV`) sobre o Random Forest, avaliada por validacao cruzada de 10 folds, para identificar a configuracao mais adequada a este problema especifico.

In [ ]:
param_dist = {
    'n_estimators': [100, 150, 200, 300],
    'max_features': [20, 50, 80, 'sqrt'],
    'max_depth': [None, 10, 20, 30],
    'min_samples_leaf': [1, 2, 4]
}
search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_dist, n_iter=30, cv=10, scoring='accuracy',
    n_jobs=-1, random_state=42
)
search.fit(X, y)

print(f"Melhores hiperparametros: {search.best_params_}")
print(f"Acuracia (validacao cruzada): {search.best_score_:.4f}")

clf_rf_opt = search.best_estimator_

## 11. Importancia dos atributos e analise estatistica

A importancia dos atributos e extraida do Random Forest, que quantifica a contribuicao de cada descritor para a separacao das classes. Sobre os atributos mais relevantes aplica-se o teste de Kruskal-Wallis, versao nao parametrica para tres ou mais grupos, que verifica se a distribuicao do descritor difere significativamente entre as classes de potencia. Um p-valor abaixo de 0.05 indica diferenca estatisticamente significativa.

In [ ]:
fi_df = pd.DataFrame({'Feature': X.columns, 'Importance': clf1.feature_importances_})
fi_df = fi_df.sort_values('Importance', ascending=False).reset_index(drop=True)
top_features = fi_df.head(21)['Feature'].tolist()
print("Top 21 atributos por importancia:")
print(fi_df.head(21).to_string(index=False))

results = []
for feat in top_features:
    v0, v1, v2 = X.loc[y == 0, feat], X.loc[y == 1, feat], X.loc[y == 2, feat]
    stat, p = kruskal(v0, v1, v2)
    results.append({'Feature': feat, 'H-stat': round(stat, 3),
                    'p-value': f"{p:.2e}", 'Significativo': 'Sim' if p < 0.05 else 'Nao'})
kruskal_df = pd.DataFrame(results)
kruskal_df.to_csv(r'..\data\figures\model2_kruskal.csv', index=False)
print()
print(kruskal_df.to_string(index=False))

## 12. Distribuicao dos principais descritores por classe

Os graficos de violino mostram como os quatro descritores mais importantes se distribuem entre as tres classes de potencia. Diferencas visiveis na forma e na posicao das distribuicoes confirmam que o modelo se apoia em padroes quimicos reais, e nao em ruido.

In [ ]:
plot_df = x_train.copy()
plot_df['Class'] = y_train.values

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, feat in zip(axes.flatten(), top_features[:4]):
    sns.violinplot(data=plot_df, x='Class', y=feat, palette='rocket', ax=ax)
    ax.set_xticklabels(['Classe 0', 'Classe 1', 'Classe 2'])
    ax.set_xlabel('')
    ax.set_title(feat, fontweight='bold')
plt.suptitle('Distribuicao dos principais descritores por classe de potencia', fontsize=14)
plt.tight_layout()
plt.savefig(r'..\data\figures\model2_violins.png', dpi=300, bbox_inches='tight')
plt.show()

## 13. Persistencia do modelo

O Random Forest otimizado, melhor modelo segundo a validacao cruzada, e salvo junto da lista de atributos mais relevantes. Esses arquivos sao reutilizados na aplicacao interativa que integra os tres modelos do projeto.

In [ ]:
os.makedirs('../models', exist_ok=True)

with open(r'..\models\HGB_model_potency.pkl', 'wb') as f:
    pickle.dump(clf_rf_opt, f)

with open(r'..\models\top_features_model2.pkl', 'wb') as f:
    pickle.dump(top_features, f)

print("Modelo de potencia e atributos salvos em ../models/")

## 14. Conclusoes

**Desempenho dos modelos (validacao cruzada de 10 folds):**

| Modelo | Acuracia (CV) |
|---|---|
| Random Forest otimizado | aprox. 0.76 |
| HistGradientBoosting | aprox. 0.73 |
| XGBoost | aprox. 0.70 |
| SVM (RBF) | aprox. 0.72 |

**Principais observacoes:**

- A classificacao de potencia em tres niveis e intrinsecamente mais dificil que a distincao entre inibidor e nao inibidor (tratada no notebook `03_modelo_classificacao1`), pois exige discriminar graus de atividade dentro do mesmo grupo de moleculas ativas.

- A padronizacao foi decisiva para o SVM: sem `StandardScaler`, o modelo colapsava para a classe majoritaria; com padronizacao, passou a distribuir as predicoes corretamente entre as tres classes.

- A otimizacao de hiperparametros elevou o desempenho do Random Forest, tornando-o o melhor modelo do conjunto na metrica mais robusta (validacao cruzada).

- Com um conjunto de teste de aproximadamente 100 amostras, a validacao cruzada e a metrica de referencia mais confiavel, por reduzir a variancia associada a uma unica particao.

- O teste de Kruskal-Wallis confirmou que os descritores mais importantes apresentam distribuicoes significativamente distintas entre as classes, sustentando a validade quimica do modelo.

**Proxima etapa:** modelo de regressao para estimativa continua do valor de IC50.